# 💻 Notebook do Aluno — Aula 09: Agentes de IA ReAct, tools e function calling

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 09/14 — Módulo 3: Interfaces, Agentes e Integração**  
**⏱️ 1h40min**  
**🤖 ReAct · @tool · AgentExecutor**  
**🔁 Andaime 50%**  

---

## 🎯 Objetivo da aula

Entender o que diferencia um agente de uma chain. Construir um agente com 3 tools que decide autonomamente qual ferramenta usar para cada pergunta — com o loop de raciocínio visível via verbose=True.

---

## Como usar este notebook

- Rode as células **na ordem**, de cima para baixo (`Shift+Enter`).
- Complete apenas as partes marcadas com `___` e `👉 LACUNA`.
- Não apague o código já pronto — ele é o andaime do lab.
- Salve sua cópia: **Arquivo > Salvar uma cópia no Drive**.

## 📋 Roteiro do Lab

**Lab — Aula 09 · 2º Semestre**  
### Agente com 3 tools do domínio ★★★

*Grupo 3–4 · 25 minutos · Google Colab*

1. Complete as 4 lacunas — descriptions das 3 tools (a parte mais importante) e parâmetros do AgentExecutor.
2. Teste os 4 cenários — RAG, Web, Calculadora e multi-step. Verifique no log verbose se o agente escolheu a tool correta em cada caso.
3. Análise do roteamento: ajuste uma description que fez o agente errar a tool. Compare o log antes e depois da correção.
4. Limite do agente: faça uma pergunta ambígua que poderia ir para o RAG OU para a web. Veja o que o agente escolhe e explique por quê (com base no Thought logado).

> **🎯 Gabarito das lacunas**
>
> Lacuna 1: "Use quando o usuário perguntar sobre informações dos documentos do domínio... NÃO use para busca na web nem para cálculos."
>
> Lacuna 2: "Use quando precisar de informações atuais não presentes nos documentos... NÃO use para conteúdo interno do domínio."
>
> Lacuna 3: "Use para cálculos matemáticos. Recebe expressão Python válida (ex: '24*30'). NÃO use para buscar informações."
>
> Lacuna 4: tools=[buscar_nos_documentos, buscar_na_web, calcular], verbose=True, max_iterations=5

---

## 🧩 Notebook Aluno — 50% de lacunas

Complete as lacunas marcadas com `___`.

In [ ]:
!pip install langchain langchain-ollama langchain-community duckduckgo-search chromadb pymupdf -q

from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_ollama import ChatOllama
from langchain.agents import AgentExecutor, create_react_agent
from langchain import hub
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
llm = ChatOllama(model="gpt-oss:120b", temperature=0)

# Retriever do CKP02 já indexado
db        = Chroma(persist_directory="/content/ckp02", embedding_function=embeddings)
retriever = db.as_retriever(search_kwargs={"k":3})

# 👉 LACUNA 1: escreva a description da tool de busca nos documentos
@tool
def buscar_nos_documentos(query: str) -> str:
    """___"""  # escreva: quando usar, quando NÃO usar, o que retorna
    docs = retriever.invoke(query)
    if not docs: return "Nenhum documento relevante encontrado."
    return "\n\n".join(f"[pág.{d.metadata.get('page',0)+1}] {d.page_content}" for d in docs)

# 👉 LACUNA 2: escreva a description da tool de busca na web
@tool
def buscar_na_web(query: str) -> str:
    """___"""  # quando usar vs. quando NÃO usar (ex: não para docs internos)
    return DuckDuckGoSearchRun().run(query)

# 👉 LACUNA 3: escreva a description da calculadora
@tool
def calcular(expressao: str) -> str:
    """___"""  # mencionar que recebe expressão Python e retorna número
    try: return str(eval(expressao,{"__builtins__":{}},{}))
    except Exception as e: return f"Erro: {e}"

# 👉 LACUNA 4: monte o AgentExecutor com verbose=True e max_iterations=5
agente   = create_react_agent(llm, [buscar_nos_documentos, buscar_na_web, calcular], hub.pull("hwchase17/react"))
executor = AgentExecutor(agent=agente, tools=[___], verbose=___, max_iterations=___, handle_parsing_errors=True)

# Testar com 4 perguntas que exercitam cada cenário
for q in [
    "Qual é a cláusula de garantia no documento?",         # → RAG
    "Qual é o dólar hoje?",                               # → Web
    "Quanto é 450 * 1.12?",                               # → Calc
    "Qual o prazo de garantia em dias (meses × 30)?",     # → RAG + Calc
]:
    print(f"\n{'='*50}\nPergunta: {q}")
    print(executor.invoke({"input":q})["output"])

---

## ✍️ Suas anotações

Registre aqui as observações pedidas no roteiro (qualidade dos resultados, comparações e conclusões do grupo).

## 📚 Referências da aula

- Paper Yao, S. et al. — "ReAct: Synergizing Reasoning and Acting in Language Models." ICLR, 2023. O paper original do padrão ReAct. arxiv.org/abs/2210.03629
- Blog Anthropic Engineering — "Building Effective Agents" (2025). A referência desta aula para quando usar workflow vs. agente. anthropic.com/engineering/building-effective-agents
- Docs LangChain — AgentExecutor, create_react_agent, @tool decorator. python.langchain.com/docs/how_to/agent_executor
- Segurança OWASP LLM Top 10 — LLM01: Prompt Injection. Documentação de riscos de segurança em sistemas LLM. owasp.org/www-project-top-10-for-large-language-model-applications
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 2 — Agentes inteligentes: o modelo percepção-ação que fundamenta o loop agêntico desta aula.
- Ebook Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 18: Guardrails/Safety Patterns — as seis camadas de defesa por trás do guardrail de prompt injection desta aula.

---

**Próxima Aula — Aula 10 · 19/10** — Context Engineering para agentes — curadoria em loop agêntico
  
Curar o contexto em loop. RAG como tool oficial. Memory seletiva entre sessões.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*